In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

loader = PyPDFLoader('./data/data_science_syllabus.pdf')
docs = loader.load()
len(docs) #10

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

embedding_model = OpenAIEmbeddings(model = 'text-embedding-3-large')

vector_store = Chroma.from_documents(
    documents = splitted_data,
    embeddings = embedding_model
)

query = 'Machine Learning and Data Science Content'
data = vector_store.similarity_search(query=query)
len(data), data[0]

context = ''
for doc in data:
    context += doc.page_content + '\n'
print(context) #This context will be provided to LLM

llm = ChatOpenAI(model ='gpt-5')
result = llm.invoke(f"""
        Can you provide me the answer based on provided context for my question?:
        context:{context}, question = {query}""")

#Here instead of providing entire content of PDF, we use:
    #pyPDFLoader to load the document
    #splitter to split the data into list format with size and overlap
    #vectorstore stored the data into vector format and we perform similarity_search
    #on it, that provides us the exact context required which we pass to ChatOpenAI

In [ ]:
#Proper Project:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

loader = PyPDFLoader('./data/data_science_syllabus.pdf')
docs = loader.load()
len(docs) #10

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

embedding_model = OpenAIEmbeddings(model = 'text-embedding-3-large')

vertor_store = Chroma.from_documents(
    documents = splitted_data,
    embeddings = embedding_model
)

query = 'Machine Learning and Data Science Content'
data = vertor_store.similarity_search(query=query)
len(data), data[0]

llm = ChatOpenAI(model ='gpt-5')

#Chain - Context_generate | prompt | llm | strparser

def get_context(query:str):
    data = vertor_store.similarity_search(query=query)
    context = '' 
    #This context will be provided to LLM
    for doc in data:
        context += doc.page_content + '\n'

    return {
        'context': context,
        'question': query
    }

prompt = PromptTemplate.from_template('''
    You are a helpful assistant provide ans based on context for user ques
    incase you don't know the ans , mention i don't know
    context: {context}
    Question: {question}''')

rag_chain = get_context | prompt | llm
res = rag_chain.invoke('What is the duration of my course?')
print(res.content)
